# Lab 2: Write Prepared Data to Iceberg Tables

**Persona:** Data Engineer &nbsp;|&nbsp; **Lab:** 2 - Data Prep (Common Step)

## Overview

This notebook is the **shared final step** after any Lab 2B–2E data prep notebook. It reads the processed `train.csv` and `test.csv` from S3 and writes them into pre-provisioned Apache Iceberg tables in two layers:

1. **S3 Table Bucket** — AWS-managed Iceberg storage
2. **Default Glue Catalog Iceberg tables** — queryable by Athena, QuickSight, and Lab 5 monitoring

### Infrastructure is pre-provisioned

The table bucket, namespace, Glue database, and both Iceberg tables (`training_data`, `evaluation_data`) are created by the **`5-data-prep.yaml`** CloudFormation template during account provisioning. This notebook **only writes data** — it does not create any infrastructure or manage permissions.

```
CloudFormation (5-data-prep.yaml)          This notebook
  ├─ S3 Table Bucket                          │
  ├─ Namespace                                │  reads train.csv / test.csv
  ├─ Glue database (bank_marketing)           │  → overwrite training_data
  └─ Iceberg tables (empty schemas)           │  → overwrite evaluation_data
```

### Prerequisites

- The **`5-data-prep.yaml`** stack has been deployed (creates the Iceberg infrastructure)
- Completed **one** of: Lab 2B, 2C, 2D, or 2E — processed CSVs exist at:
  - `s3://<default-bucket>/bank-marketing-lab/data/train/train.csv`
  - `s3://<default-bucket>/bank-marketing-lab/data/test/test.csv`

### What this enables downstream

- **Lab 3**: Reads training data from Athena for model building
- **Lab 5**: Uses Iceberg snapshot IDs (`FOR VERSION AS OF`) as frozen baselines for drift detection
- **QuickSight**: Connects to the default catalog tables for governance dashboards

---

## Section 1: Setup

In [ ]:
# Install dependencies
%pip install "pyiceberg[pandas,pyarrow,glue]" "sagemaker>=3.17,<4" --quiet

In [ ]:
import boto3
import os
import sagemaker
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pcsv
import time
from datetime import datetime
from pyiceberg.catalog import load_catalog
from sagemaker.core.helper.session_helper import Session, get_execution_role

# Initialize session
sagemaker_session = Session()
region = sagemaker_session.boto_region_name
role = get_execution_role()
bucket = sagemaker_session.default_bucket()

sts_client = boto3.client('sts', region_name=region)
account_id = sts_client.get_caller_identity()['Account']

# --- Resource names (single source of truth) ---
# lab0-setup/setup.ipynb discovered these once and wrote them to the repo-root
# .env. The shared workshop_env loader reads that file — no lab hardcodes or
# re-declares project/schema names. (Repo root holds workshop_env.py; find it
# by walking up, then import the one shared loader.)
import sys
for _c in [os.getcwd(), *[str(p) for p in __import__('pathlib').Path(os.getcwd()).parents]]:
    if os.path.exists(os.path.join(_c, 'workshop_env.py')):
        sys.path.insert(0, _c); break
from workshop_env import load_workshop_env
_env = load_workshop_env()
PROJECT_NAME = _env['PROJECT_NAME']
ATHENA_DATABASE = _env['ATHENA_DATABASE']
TRAINING_TABLE = _env['TRAINING_TABLE']
EVALUATION_TABLE = _env['EVALUATION_TABLE']
S3T_NAMESPACE = _env['S3T_NAMESPACE']
prefix = _env['DATA_PREFIX']
TABLE_BUCKET_NAME = f'{PROJECT_NAME}-monitoring-{account_id}'  # S3 Table Bucket created by the template

# S3 paths (output of Lab 2B/2C/2D/2E)
train_s3 = f's3://{bucket}/{prefix}/data/train/train.csv'
test_s3 = f's3://{bucket}/{prefix}/data/test/test.csv'

print(f'Region:          {region}')
print(f'Account:         {account_id}')
print(f'S3 Bucket:       {bucket}')
print(f'Athena database: {ATHENA_DATABASE}')
print(f'S3 Table Bucket: {TABLE_BUCKET_NAME}')
print(f'Train CSV:       {train_s3}')
print(f'Test CSV:        {test_s3}')

In [ ]:
# Verify the pre-provisioned infrastructure exists (fail early with a clear message)
glue = boto3.client('glue', region_name=region)
s3tables = boto3.client('s3tables', region_name=region)

errors = []

# 1. Glue database + tables
try:
    glue.get_database(Name=ATHENA_DATABASE)
    for t in (TRAINING_TABLE, EVALUATION_TABLE):
        glue.get_table(DatabaseName=ATHENA_DATABASE, Name=t)
    print(f'✓ Glue database and tables exist: {ATHENA_DATABASE}.{{{TRAINING_TABLE}, {EVALUATION_TABLE}}}')
except Exception as e:
    errors.append(f'Glue: {e}')

# 2. S3 Table Bucket
try:
    table_bucket_arn = f'arn:aws:s3tables:{region}:{account_id}:bucket/{TABLE_BUCKET_NAME}'
    s3tables.get_table_bucket(tableBucketARN=table_bucket_arn)
    print(f'✓ S3 Table Bucket exists: {TABLE_BUCKET_NAME}')
except Exception as e:
    errors.append(f'S3 Tables: {e}')

if errors:
    raise RuntimeError(
        'Pre-provisioned infrastructure not found. Ensure the 5-data-prep.yaml stack '
        'is deployed and PROJECT_NAME matches the deploy-time ProjectName.\n  '
        + '\n  '.join(errors)
    )

# 3. S3 Tables integration with AWS analytics services (needed by Section 3 only).
#    Enabling it registers a federated Glue catalog named
#    's3tablescatalog/<table-bucket>' and hands Lake Formation control of the
#    table bucket's managed storage. That is an account-level setup step, so it
#    may legitimately be off — Section 3 checks this flag and skips itself
#    rather than failing with an opaque 'Catalog not found'.
S3T_CATALOG_ID = f's3tablescatalog/{TABLE_BUCKET_NAME}'
try:
    glue.get_catalog(CatalogId=S3T_CATALOG_ID)
    S3TABLES_INTEGRATION = True
    print(f'✓ S3 Tables analytics integration enabled: catalog {S3T_CATALOG_ID}')
except glue.exceptions.EntityNotFoundException:
    S3TABLES_INTEGRATION = False
    print(f'⊘ S3 Tables analytics integration NOT enabled (no Glue catalog {S3T_CATALOG_ID})')
    print('  Section 3 will be skipped; Section 4 (what Lab 3A and Lab 5 read) still runs.')
except Exception as e:
    # Can't tell either way (e.g. glue:GetCatalog not granted) — let Section 3 try.
    S3TABLES_INTEGRATION = None
    print(f'? Could not check the S3 Tables analytics integration: {e}')

print('\n✓ All required infrastructure present — ready to write data.')

---

## Section 2: Read Processed CSVs and Prepare for Iceberg

The CSVs from Lab 2B–2E are headerless with the target as column 0. We read them, assign the schema's column names, and add the synthetic columns (identifier, timestamp) that the pre-provisioned tables expect.

In [ ]:
# Feature columns come from the single shared schema (workshop_common reads
# lab5-monitoring/src/config/dataset_schema.yaml) — no hardcoded feature list.
from workshop_common import schema as _schema
FEATURE_COLUMNS = _schema.feature_names()

def read_and_prepare(s3_uri, split_name):
    """Read headerless CSV from S3, apply the canonical schema."""
    _, _, rest = s3_uri.partition('s3://')
    b, _, key = rest.partition('/')
    s3_client = boto3.client('s3', region_name=region)
    obj = s3_client.get_object(Bucket=b, Key=key)
    
    csv_columns = ['y'] + FEATURE_COLUMNS
    read_opts = pcsv.ReadOptions(column_names=csv_columns)
    df = pcsv.read_csv(obj['Body'], read_options=read_opts).to_pandas()
    
    # Synthetic identifier
    df['client_id'] = [f'bm-{split_name}-{i}' for i in range(len(df))]
    
    # Synthetic timestamp (timezone-naive to match Iceberg 'timestamp')
    base_ts = datetime(2024, 1, 1)
    df['contact_timestamp'] = pd.to_datetime(
        [base_ts + pd.Timedelta(hours=i) for i in range(len(df))]
    )
    
    # Target: int 0/1 -> boolean
    df['subscribed'] = df['y'].astype(bool)
    
    # Placeholder columns (filled by Lab 3 inference)
    df['prediction'] = None
    df['probability_positive'] = None
    
    final_cols = (
        ['client_id'] + FEATURE_COLUMNS +
        ['contact_timestamp', 'prediction', 'probability_positive', 'subscribed']
    )
    return df[final_cols]

train_df = read_and_prepare(train_s3, 'train')
test_df = read_and_prepare(test_s3, 'test')

print(f'Train: {train_df.shape[0]} rows, {train_df.shape[1]} cols')
print(f'Test:  {test_df.shape[0]} rows, {test_df.shape[1]} cols')
train_df.head(3)

In [ ]:
# PyArrow schema must match the pre-provisioned table schema exactly.
# contact_timestamp is timezone-NAIVE to match the template's TIMESTAMP / Iceberg 'timestamp'.
iceberg_schema = pa.schema([
    pa.field('client_id', pa.string()),
    *[pa.field(col, pa.float64()) for col in FEATURE_COLUMNS],
    pa.field('contact_timestamp', pa.timestamp('us')),
    pa.field('prediction', pa.bool_()),
    pa.field('probability_positive', pa.float64()),
    pa.field('subscribed', pa.bool_()),
])

train_arrow = pa.Table.from_pandas(train_df, schema=iceberg_schema, preserve_index=False)
test_arrow = pa.Table.from_pandas(test_df, schema=iceberg_schema, preserve_index=False)

print(f'✓ PyArrow tables ready ({len(iceberg_schema)} columns)')
print(f'  Train: {train_arrow.num_rows} rows')
print(f'  Test:  {test_arrow.num_rows} rows')

---

## Section 3: Write to the S3 Table Bucket (Managed Iceberg)

The table bucket, namespace, and tables already exist (created by the template). We load the existing tables and **overwrite** them with the prepared data.

> **Requires the S3 Tables integration with AWS analytics services.** Writing to a
> table bucket goes through the Glue Iceberg REST endpoint, which only exists once
> that account-level integration is enabled (it registers the federated
> `s3tablescatalog/<table-bucket>` catalog in Glue). Section 1 detects this; if the
> integration is off, this section prints what to enable and skips itself. Section 4
> is the write that Lab 3A and Lab 5 depend on, and it runs either way.

In [ ]:
# Write to the S3 Table Bucket through the Glue Iceberg REST endpoint.
#
# Why the Glue endpoint and not https://s3tables.<region>.amazonaws.com/iceberg:
# both speak the Iceberg REST protocol, but a table bucket's managed storage
# (the '<table-id>--table-s3' bucket) is not writable with the caller's own
# credentials, and the S3 Tables endpoint vends none. The Glue endpoint, backed
# by the analytics-services integration, is the path that can actually commit.
if S3TABLES_INTEGRATION is False:
    print(f'⊘ Skipped — no Glue catalog {S3T_CATALOG_ID}.')
    print()
    print('  The S3 Tables integration with AWS analytics services is a one-time,')
    print('  account-level setup step (it needs glue:CreateCatalog plus Lake')
    print('  Formation permissions, which the lab roles deliberately do not have):')
    print('    Console: S3 → Table buckets → Enable integration')
    print('    API:     glue:CreateCatalog with the built-in aws:s3tables connection')
    print()
    print('  Section 4 writes the same two datasets to the default-catalog Iceberg')
    print('  tables. That is what Lab 3A trains from and Lab 5 monitors, so nothing')
    print('  downstream depends on this section.')
else:
    s3t_catalog = load_catalog(
        's3tablescatalog',
        **{
            'type': 'rest',
            'warehouse': f'{account_id}:s3tablescatalog/{TABLE_BUCKET_NAME}',
            'uri': f'https://glue.{region}.amazonaws.com/iceberg',
            'rest.sigv4-enabled': 'true',
            'rest.signing-name': 'glue',
            'rest.signing-region': region,
        }
    )

    # Load the pre-provisioned tables and overwrite with data
    for table_name, arrow_data in [
        (TRAINING_TABLE, train_arrow),
        (EVALUATION_TABLE, test_arrow),
    ]:
        tbl = s3t_catalog.load_table(f'{S3T_NAMESPACE}.{table_name}')
        tbl.overwrite(arrow_data)
        print(f'✓ Wrote {arrow_data.num_rows} rows to S3 Table {S3T_NAMESPACE}.{table_name}')

    print(f'\n✓ S3 Table Bucket populated')
    print(f'  Athena query: SELECT * FROM "s3tablescatalog/{TABLE_BUCKET_NAME}"."{S3T_NAMESPACE}"."{TRAINING_TABLE}" LIMIT 5;')

---

## Section 4: Write to the Default Catalog Iceberg Tables

These tables (Athena / QuickSight / Lab 5) also already exist. Load them via the Glue catalog and overwrite.

In [ ]:
# Connect to the default Glue catalog and overwrite the pre-provisioned tables
glue_catalog = load_catalog('glue', **{'type': 'glue', 'client.region': region})

training_table = glue_catalog.load_table(f'{ATHENA_DATABASE}.{TRAINING_TABLE}')
evaluation_table = glue_catalog.load_table(f'{ATHENA_DATABASE}.{EVALUATION_TABLE}')

training_table.overwrite(train_arrow)
print(f'✓ Wrote {train_arrow.num_rows} rows to {ATHENA_DATABASE}.{TRAINING_TABLE}')

evaluation_table.overwrite(test_arrow)
print(f'✓ Wrote {test_arrow.num_rows} rows to {ATHENA_DATABASE}.{EVALUATION_TABLE}')

---

## Section 5: Capture Snapshot IDs

Iceberg snapshot IDs are the immutable references Lab 5's drift monitor uses to time-travel to the exact data the model was trained on. Lab 3 records these in `baseline.json`.

In [ ]:
training_snapshot_id = training_table.snapshots()[-1].snapshot_id
evaluation_snapshot_id = evaluation_table.snapshots()[-1].snapshot_id

print('✓ Iceberg Snapshot IDs (record these in baseline.json for Lab 5):')
print(f'  training_snapshot_id:   {training_snapshot_id}')
print(f'  evaluation_snapshot_id: {evaluation_snapshot_id}')
print(f'\n  Time-travel example:')
print(f'    SELECT * FROM {ATHENA_DATABASE}.{TRAINING_TABLE}')
print(f'    FOR VERSION AS OF {training_snapshot_id} LIMIT 5;')

---

## Section 6: Verify

In [ ]:
# Verify via Athena
athena_client = boto3.client('athena', region_name=region)
ATHENA_OUTPUT = f's3://{bucket}/{prefix}/athena-results/'

def run_athena_query(sql, database=None):
    params = {'QueryString': sql, 'ResultConfiguration': {'OutputLocation': ATHENA_OUTPUT}}
    if database:
        params['QueryExecutionContext'] = {'Database': database}
    qid = athena_client.start_query_execution(**params)['QueryExecutionId']
    while True:
        state = athena_client.get_query_execution(QueryExecutionId=qid)['QueryExecution']['Status']['State']
        if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
            break
        time.sleep(2)
    if state != 'SUCCEEDED':
        reason = athena_client.get_query_execution(QueryExecutionId=qid)['QueryExecution']['Status'].get('StateChangeReason', 'Unknown')
        raise RuntimeError(f'Athena query {state}: {reason}')
    return qid

for table_name in (TRAINING_TABLE, EVALUATION_TABLE):
    qid = run_athena_query(f'SELECT COUNT(*) AS cnt FROM {ATHENA_DATABASE}.{table_name}', database=ATHENA_DATABASE)
    cnt = athena_client.get_query_results(QueryExecutionId=qid)['ResultSet']['Rows'][1]['Data'][0]['VarCharValue']
    print(f'✓ {ATHENA_DATABASE}.{table_name}: {cnt} rows')

# Preview ALL columns for both tables. pandas is used only to render the
# full wide result cleanly (Athena returns every column via SELECT *).
import pandas as pd
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', None)

def preview_all_columns(table_name, limit=5):
    qid = run_athena_query(
        f'SELECT * FROM {ATHENA_DATABASE}.{table_name} LIMIT {limit}',
        database=ATHENA_DATABASE
    )
    rows = athena_client.get_query_results(QueryExecutionId=qid)['ResultSet']['Rows']
    header = [d.get('VarCharValue', 'NULL') for d in rows[0]['Data']]
    data = [[d.get('VarCharValue', 'NULL') for d in r['Data']] for r in rows[1:]]
    return pd.DataFrame(data, columns=header)

for table_name in (TRAINING_TABLE, EVALUATION_TABLE):
    print(f'\n--- Preview: {table_name} (all {len(iceberg_schema.names)} columns) ---')
    display(preview_all_columns(table_name))

---

## Summary

You wrote the processed bank marketing datasets into the pre-provisioned Iceberg tables:

### S3 Table Bucket (managed Iceberg)
- Bucket: `<project>-monitoring-<account-id>` (created by `5-data-prep.yaml`)
- Namespace: `bank_marketing`
- Tables: `training_data`, `evaluation_data`

### Default Catalog Iceberg (Athena / QuickSight / Lab 5)
- Database: `bank_marketing`
- Tables: `training_data`, `evaluation_data`
- Snapshot IDs captured for drift monitoring baseline pinning

### What changed vs. the self-provisioning notebook

This notebook assumes **`5-data-prep.yaml`** already created the table bucket, namespace, Glue database, and empty Iceberg tables. It only **reads CSVs and overwrites the tables** — no resource creation, no Lake Formation grants. That work now happens once, at account provisioning time.

### Next Steps

- **Lab 3A**: Train XGBoost reading from the Athena Iceberg tables
- **Lab 5**: Drift monitoring uses the snapshot IDs as frozen baselines
- **QuickSight**: Governance dashboard reads from the default catalog tables